In [ ]:
%pwd

LOAD PDF

In [ ]:
from pathlib import Path
from langchain_community.document_loaders import PyPDFLoader, DirectoryLoader


def load_pdf_file(data):
    loader = DirectoryLoader(
        data,
        glob="*.pdf",
        loader_cls=PyPDFLoader
    )
    documents = loader.load()
    return documents


PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "research" else Path.cwd()
DATA_DIR = PROJECT_ROOT / "data"

print("Project root:", PROJECT_ROOT)
print("Data folder:", DATA_DIR)

extracted_data = load_pdf_file(str(DATA_DIR))
print("Pages loaded:", len(extracted_data))

CLEAN METADATA

In [ ]:
from langchain_core.documents import Document


def filter_to_minimal_docs(docs):
    minimal_docs = [
        Document(
            page_content=doc.page_content,
            metadata={"source": doc.metadata.get("source")}
        )
        for doc in docs
    ]
    return minimal_docs


filter_data = filter_to_minimal_docs(extracted_data)
print("Filtered documents:", len(filter_data))

SPLIT INTO CHUNKS

In [ ]:
from langchain.text_splitter import RecursiveCharacterTextSplitter


def text_split(extracted_data):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=500,
        chunk_overlap=20
    )
    text_chunks = text_splitter.split_documents(extracted_data)
    return text_chunks


text_chunks = text_split(filter_data)
print("Length of text chunks:", len(text_chunks))

DOWNLOAD HUGGINGFACE EMBEDDINGS

In [ ]:
from langchain_community.embeddings import HuggingFaceEmbeddings


def download_hugging_face_embeddings():
    embeddings = HuggingFaceEmbeddings(
        model_name="sentence-transformers/all-MiniLM-L6-v2"
    )
    return embeddings


embeddings = download_hugging_face_embeddings()
query_result = embeddings.embed_query("Hello world")
print("Embedding dimension:", len(query_result))

LOAD API KEYS

In [ ]:
from dotenv import load_dotenv
import os


load_dotenv()

PINECONE_API_KEY = os.environ.get("PINECONE_API_KEY")
OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY")

os.environ["PINECONE_API_KEY"] = PINECONE_API_KEY
os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY

print("Pinecone key loaded:", bool(PINECONE_API_KEY))
print("OpenAI key loaded:", bool(OPENAI_API_KEY))

CONNECT TO PINECONE AND CREATE INDEX

In [ ]:
from pinecone import Pinecone, ServerlessSpec


pc = Pinecone(api_key=PINECONE_API_KEY)

index_name = "medical-chatbot"

if not pc.has_index(index_name):
    pc.create_index(
        name=index_name,
        dimension=384,
        metric="cosine",
        spec=ServerlessSpec(
            cloud="aws",
            region="us-east-1"
        )
    )
    print("Index created")
else:
    print("Index already exists")

index = pc.Index(index_name)

UPLOAD DATA TO PINECONE

In [ ]:
from langchain_pinecone import PineconeVectorStore


# This guard prevents duplicate uploads if you run this notebook again.
stats = index.describe_index_stats()
if hasattr(stats, "total_vector_count"):
    vector_count = stats.total_vector_count
else:
    vector_count = stats.get("total_vector_count", 0)

if vector_count == 0:
    docsearch = PineconeVectorStore.from_documents(
        documents=text_chunks,
        index_name=index_name,
        embedding=embeddings
    )
    print("Data uploaded to Pinecone")
else:
    print(f"Pinecone already has {vector_count} vectors. Skipping upload.")

LOAD EXISTING PINECONE INDEX

In [ ]:
docsearch = PineconeVectorStore.from_existing_index(
    index_name=index_name,
    embedding=embeddings
)

print("docsearch loaded:", docsearch)

DIRECT PINECONE QUERY CHECK

In [ ]:
QUESTION = "What is fever?"
TOP_K = 3


def search_pinecone(question, top_k=3):
    query_vector = embeddings.embed_query(question)

    results = index.query(
        vector=query_vector,
        top_k=top_k,
        include_metadata=True
    )

    matches = results.get("matches", [])

    print("Question:", question)
    print("Relevant chunks from Pinecone:", len(matches))

    for i, match in enumerate(matches, 1):
        metadata = match.get("metadata", {})
        text = metadata.get("text", "")
        source = metadata.get("source", "")

        print(f"\n--- Pinecone Chunk {i} ---")
        print("ID:", match.get("id"))
        print("Score:", match.get("score"))
        print("Source:", source)
        print("Text:")
        print(text[:1000])

    return matches


pinecone_matches = search_pinecone(QUESTION, TOP_K)

TEST RETRIEVAL

In [ ]:
retriever = docsearch.as_retriever(
    search_type="similarity",
    search_kwargs={"k": TOP_K}
)

retrieved_docs = retriever.invoke(QUESTION)

print("Number of results:", len(retrieved_docs))

for i, doc in enumerate(retrieved_docs, 1):
    print(f"\n--- Result {i} ---")
    print(doc.page_content[:700])
    print(doc.metadata)

DIAGNOSTIC CHECK

In [ ]:
print("CWD:", os.getcwd())
print("Project root:", PROJECT_ROOT)
print("Data folder exists:", DATA_DIR.exists())
print("Files in data:", os.listdir(DATA_DIR) if DATA_DIR.exists() else "NOT FOUND")
print("Pinecone index stats:", index.describe_index_stats())